# Triton Inference Server Administration

This notebook provides administrative capabilities for managing the Triton Inference Server deployment:

1. **Scale Deployment** - Adjust the number of Triton server replicas
2. **Resource Management** - View and update CPU/Memory/GPU resources
3. **Model Management** - Load, Unload, Pin, and Cordon models
4. **Metrics & Monitoring** - View GPU/CPU utilization and memory usage

## Prerequisites

- Set the appropriate environment variables for your deployment
- Ensure you have the required permissions (admin access for scaling/resources)

In [ ]:
# Install required packages if needed
# !pip install requests pandas tabulate

In [ ]:
import os
import json
import requests
from typing import Optional, Dict, Any, List
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# Try to import optional packages for nicer display
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("Note: Install pandas for nicer table display: pip install pandas")

## Configuration

Configure the endpoints for your Triton deployment. Update these values based on your environment.

In [ ]:
@dataclass
class TritonConfig:
    """Configuration for Triton endpoints."""
    # Proxy URL for model management (Load/Unload/Pin/Cordon) and metrics
    proxy_url: str = "http://triton-inference-server-proxy.domino-inference-dev.svc.cluster.local:8080"
    
    # Admin API URL for Kubernetes operations (Scale/Resources)
    admin_url: str = "http://triton-inference-server-admin.domino-inference-dev.svc.cluster.local:8000"
    
    # Authentication token (fetched from Domino API proxy)
    api_key: Optional[str] = None
    
    def __post_init__(self):
        if self.api_key is None:
            self.api_key = self._get_domino_access_token()
    
    def _get_domino_access_token(self) -> Optional[str]:
        """Fetch access token from Domino API proxy."""
        api_proxy = os.environ.get("DOMINO_API_PROXY")
        if api_proxy:
            try:
                response = requests.get(f"{api_proxy}/access-token", timeout=5)
                response.raise_for_status()
                return response.text.strip()
            except requests.exceptions.RequestException as e:
                print(f"Warning: Failed to fetch access token from Domino API proxy: {e}")
        
        # Fallback to environment variable
        return os.environ.get("DOMINO_USER_API_KEY")
    
    @property
    def headers(self) -> Dict[str, str]:
        """Get headers for API requests."""
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["X-Domino-Api-Key"] = self.api_key
            headers["Authorization"] = f"Bearer {self.api_key}"
        return headers


# Initialize configuration - Update these for your environment
config = TritonConfig(
    proxy_url=os.environ.get("TRITON_PROXY_URL", "http://triton-inference-server-proxy.domino-inference-dev.svc.cluster.local:8080"),
    admin_url=os.environ.get("TRITON_ADMIN_URL", "http://triton-inference-server-admin.domino-inference-dev.svc.cluster.local:8000"),
)

print(f"Proxy URL: {config.proxy_url}")
print(f"Admin URL: {config.admin_url}")
print(f"API Key configured: {'Yes' if config.api_key else 'No'}")
print(f"\nNote: Metrics are accessed via the proxy at {config.proxy_url}/v1/metrics/*")

## Helper Functions

In [ ]:
def api_request(method: str, url: str, headers: Dict[str, str], json_data: Any = None, timeout: int = 30) -> Dict[str, Any]:
    """Make an API request and return the response."""
    try:
        response = requests.request(
            method=method,
            url=url,
            headers=headers,
            json=json_data,
            timeout=timeout,
            verify=False  # For local development; enable in production
        )
        response.raise_for_status()
        return response.json() if response.text else {"status": "success"}
    except requests.exceptions.HTTPError as e:
        error_detail = ""
        try:
            error_detail = e.response.json()
        except:
            error_detail = e.response.text
        return {"error": str(e), "detail": error_detail, "status_code": e.response.status_code}
    except requests.exceptions.RequestException as e:
        return {"error": str(e)}


def display_json(data: Any, title: str = None):
    """Display JSON data nicely."""
    if title:
        print(f"\n{'='*60}")
        print(f" {title}")
        print(f"{'='*60}")
    print(json.dumps(data, indent=2, default=str))


def display_table(data: List[Dict], title: str = None):
    """Display data as a table."""
    if title:
        print(f"\n{'='*60}")
        print(f" {title}")
        print(f"{'='*60}")
    if HAS_PANDAS and data:
        display(pd.DataFrame(data))
    else:
        for item in data:
            print(item)

---
# 1. Scale Deployment

Scale the Triton Inference Server deployment to adjust the number of replicas.

In [ ]:
def get_deployment_status() -> Dict[str, Any]:
    """Get current deployment status including replica counts."""
    url = f"{config.admin_url}/v1/deployments/inference-server/status"
    return api_request("GET", url, config.headers)


def scale_deployment(replicas: int) -> Dict[str, Any]:
    """Scale the deployment to the specified number of replicas."""
    url = f"{config.admin_url}/v1/deployments/inference-server/scale"
    return api_request("POST", url, config.headers, json_data={"replicas": replicas})


def restart_deployment() -> Dict[str, Any]:
    """Trigger a rolling restart of the deployment."""
    url = f"{config.admin_url}/v1/deployments/inference-server/restart"
    return api_request("POST", url, config.headers)

In [ ]:
# Check current deployment status
status = get_deployment_status()
display_json(status, "Current Deployment Status")

In [ ]:
# Scale deployment (uncomment and modify to execute)
# WARNING: This will change the number of running pods!

# new_replicas = 2
# result = scale_deployment(new_replicas)
# display_json(result, f"Scaling to {new_replicas} replicas")

In [ ]:
# Restart deployment (uncomment to execute)
# WARNING: This will trigger a rolling restart!

# result = restart_deployment()
# display_json(result, "Deployment Restart")

---
# 2. Resource Management

View and update CPU, Memory, and GPU resources for the Triton deployment.

In [ ]:
def get_deployment_resources() -> Dict[str, Any]:
    """Get current resource configuration for the deployment."""
    url = f"{config.admin_url}/v1/deployments/inference-server/resources"
    return api_request("GET", url, config.headers)


def update_deployment_resources(
    cpu_request: str = None,
    cpu_limit: str = None,
    memory_request: str = None,
    memory_limit: str = None,
    gpu_request: int = None,
    gpu_limit: int = None,
    container_name: str = None
) -> Dict[str, Any]:
    """
    Update resource requests and limits for the deployment.
    
    Args:
        cpu_request: CPU request (e.g., '100m', '1', '2')
        cpu_limit: CPU limit (e.g., '500m', '2', '4')
        memory_request: Memory request (e.g., '256Mi', '1Gi', '4Gi')
        memory_limit: Memory limit (e.g., '512Mi', '2Gi', '8Gi')
        gpu_request: Number of GPUs to request
        gpu_limit: Number of GPUs to limit
        container_name: Target container (default: first container)
    """
    url = f"{config.admin_url}/v1/deployments/inference-server/resources"
    
    body = {"resources": {}}
    
    if container_name:
        body["container_name"] = container_name
    
    requests_spec = {}
    if cpu_request:
        requests_spec["cpu"] = cpu_request
    if memory_request:
        requests_spec["memory"] = memory_request
    if gpu_request is not None:
        requests_spec["gpu"] = gpu_request
    if requests_spec:
        body["resources"]["requests"] = requests_spec
    
    limits_spec = {}
    if cpu_limit:
        limits_spec["cpu"] = cpu_limit
    if memory_limit:
        limits_spec["memory"] = memory_limit
    if gpu_limit is not None:
        limits_spec["gpu"] = gpu_limit
    if limits_spec:
        body["resources"]["limits"] = limits_spec
    
    return api_request("PATCH", url, config.headers, json_data=body)


def get_pod_details() -> Dict[str, Any]:
    """Get detailed pod information."""
    url = f"{config.admin_url}/v1/deployments/inference-server/pods"
    return api_request("GET", url, config.headers)

In [ ]:
# View current resource configuration
resources = get_deployment_resources()
display_json(resources, "Current Resource Configuration")

In [ ]:
# View pod details
pods = get_pod_details()
display_json(pods, "Pod Details")

In [ ]:
# Update resources (uncomment and modify to execute)
# WARNING: This will trigger a rolling update!

# Example: Update CPU and memory
# result = update_deployment_resources(
#     cpu_request="500m",
#     cpu_limit="2",
#     memory_request="1Gi",
#     memory_limit="4Gi"
# )
# display_json(result, "Resource Update Result")

# Example: Request a GPU
# result = update_deployment_resources(
#     gpu_request=1,
#     gpu_limit=1
# )
# display_json(result, "GPU Resource Update Result")

---
# 3. Model Management

Load, Unload, Pin, and Cordon models in the Triton Inference Server.

In [ ]:
def list_models() -> Dict[str, Any]:
    """List all models with their current status."""
    url = f"{config.proxy_url}/v1/dashboard/models"
    return api_request("GET", url, config.headers)


def get_model_info(model_name: str) -> Dict[str, Any]:
    """Get detailed information about a specific model."""
    url = f"{config.proxy_url}/v1/dashboard/models/{model_name}"
    return api_request("GET", url, config.headers)


def load_model(model_name: str) -> Dict[str, Any]:
    """Load a model into the inference server."""
    url = f"{config.proxy_url}/v2/repository/models/{model_name}/load"
    return api_request("POST", url, config.headers)


def unload_model(model_name: str) -> Dict[str, Any]:
    """Unload a model from the inference server."""
    url = f"{config.proxy_url}/v2/repository/models/{model_name}/unload"
    return api_request("POST", url, config.headers)


def pin_model(model_name: str) -> Dict[str, Any]:
    """Pin a model to prevent automatic eviction."""
    url = f"{config.proxy_url}/v1/models/{model_name}/pin"
    return api_request("POST", url, config.headers, json_data={"pinned": True})


def unpin_model(model_name: str) -> Dict[str, Any]:
    """Unpin a model to allow automatic eviction."""
    url = f"{config.proxy_url}/v1/models/{model_name}/pin"
    return api_request("POST", url, config.headers, json_data={"pinned": False})


def cordon_model(model_name: str) -> Dict[str, Any]:
    """Cordon a model to stop accepting new inference requests."""
    url = f"{config.proxy_url}/v1/models/{model_name}/cordon"
    return api_request("POST", url, config.headers)


def uncordon_model(model_name: str) -> Dict[str, Any]:
    """Uncordon a model to resume accepting inference requests."""
    url = f"{config.proxy_url}/v1/models/{model_name}/uncordon"
    return api_request("POST", url, config.headers)

In [ ]:
# List all models
models = list_models()

if "models" in models:
    print("\n" + "="*80)
    print(" Available Models")
    print("="*80)
    
    model_data = []
    for m in models["models"]:
        model_data.append({
            "Name": m.get("name"),
            "Loaded": "Yes" if m.get("loaded") else "No",
            "Pinned": "Yes" if m.get("pinned") else "No",
            "Cordoned": "Yes" if m.get("cordoned") else "No",
            "Access Count": m.get("access_count", 0),
            "Last Accessed By": m.get("last_accessed_by") or "N/A"
        })
    
    if HAS_PANDAS:
        display(pd.DataFrame(model_data))
    else:
        for row in model_data:
            print(row)
else:
    display_json(models, "Models Response")

In [ ]:
# Get detailed info for a specific model (uncomment and modify)

# model_name = "bert-base-uncased"
# info = get_model_info(model_name)
# display_json(info, f"Model Info: {model_name}")

In [ ]:
# Load a model (uncomment and modify)

# model_name = "bert-base-uncased"
# result = load_model(model_name)
# display_json(result, f"Load Model: {model_name}")

In [ ]:
# Unload a model (uncomment and modify)

# model_name = "bert-base-uncased"
# result = unload_model(model_name)
# display_json(result, f"Unload Model: {model_name}")

In [ ]:
# Pin a model (uncomment and modify)

# model_name = "bert-base-uncased"
# result = pin_model(model_name)
# display_json(result, f"Pin Model: {model_name}")

In [ ]:
# Unpin a model (uncomment and modify)

# model_name = "bert-base-uncased"
# result = unpin_model(model_name)
# display_json(result, f"Unpin Model: {model_name}")

In [ ]:
# Cordon a model (uncomment and modify)

# model_name = "bert-base-uncased"
# result = cordon_model(model_name)
# display_json(result, f"Cordon Model: {model_name}")

In [ ]:
# Uncordon a model (uncomment and modify)

# model_name = "bert-base-uncased"
# result = uncordon_model(model_name)
# display_json(result, f"Uncordon Model: {model_name}")

---
# 4. Metrics & Monitoring

View GPU/CPU utilization and memory usage via the HTTP proxy's metrics endpoints.

**Note:** The metrics are fetched from Triton via the HTTP proxy (`/v1/metrics/*`), since Triton's metrics port (8002) is not directly accessible from notebooks. The proxy fetches from Triton and returns parsed, structured data.

In [ ]:
def get_raw_metrics() -> str:
    """Fetch raw Prometheus metrics from Triton via the proxy."""
    url = f"{config.proxy_url}/v1/metrics"
    try:
        response = requests.get(url, headers=config.headers, timeout=10, verify=False)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        return f"Error: {e}"


def get_parsed_metrics() -> Dict[str, Any]:
    """Fetch all parsed metrics from the proxy."""
    url = f"{config.proxy_url}/v1/metrics/parsed"
    return api_request("GET", url, config.headers)


def get_gpu_metrics() -> List[Dict[str, Any]]:
    """
    Fetch GPU metrics from the proxy.
    
    Returns:
        List of GPU metrics including memory, utilization, power, and temperature.
    """
    url = f"{config.proxy_url}/v1/metrics/gpu"
    return api_request("GET", url, config.headers)


def get_cpu_memory_metrics() -> Dict[str, Any]:
    """
    Fetch CPU and memory metrics from the proxy.
    
    Returns:
        Dictionary with CPU utilization and memory usage.
    """
    url = f"{config.proxy_url}/v1/metrics/cpu"
    return api_request("GET", url, config.headers)


def get_inference_metrics() -> List[Dict[str, Any]]:
    """
    Fetch per-model inference metrics from the proxy.
    
    Returns:
        List of model inference statistics including counts and latencies.
    """
    url = f"{config.proxy_url}/v1/metrics/inference"
    return api_request("GET", url, config.headers)

In [ ]:
# Get GPU metrics from the proxy
gpu_metrics = get_gpu_metrics()

print("\n" + "="*80)
print(" GPU Metrics")
print("="*80)

if isinstance(gpu_metrics, list) and gpu_metrics:
    for gpu in gpu_metrics:
        gpu_id = gpu.get("gpu_id", "unknown")[:16]
        name = gpu.get("name", "")
        
        print(f"\n  GPU {gpu_id}" + (f" ({name})" if name else ""))
        
        # Memory
        if gpu.get("memory_total_bytes") and gpu.get("memory_used_bytes"):
            total_gb = gpu["memory_total_bytes"] / (1024**3)
            used_gb = gpu["memory_used_bytes"] / (1024**3)
            free_gb = gpu.get("memory_free_bytes", 0) / (1024**3)
            pct = (gpu["memory_used_bytes"] / gpu["memory_total_bytes"]) * 100
            print(f"    Memory: {used_gb:.2f} GB / {total_gb:.2f} GB ({pct:.1f}% used, {free_gb:.2f} GB free)")
        
        # Utilization
        if gpu.get("utilization_percent") is not None:
            print(f"    Utilization: {gpu['utilization_percent']:.1f}%")
        
        # Power
        if gpu.get("power_usage_watts") is not None:
            limit = gpu.get("power_limit_watts")
            if limit:
                print(f"    Power: {gpu['power_usage_watts']:.1f}W / {limit:.1f}W")
            else:
                print(f"    Power: {gpu['power_usage_watts']:.1f}W")
        
        # Temperature
        if gpu.get("temperature_celsius") is not None:
            print(f"    Temperature: {gpu['temperature_celsius']:.1f}C")

elif isinstance(gpu_metrics, dict) and "error" in gpu_metrics:
    print(f"  Error fetching GPU metrics: {gpu_metrics.get('error')}")
    if gpu_metrics.get("detail"):
        print(f"  Detail: {gpu_metrics.get('detail')}")
else:
    print("  No GPU metrics available (may not have a GPU or Triton is not running)")

In [ ]:
# Get CPU/Memory metrics from the proxy
cpu_mem = get_cpu_memory_metrics()

print("\n" + "="*80)
print(" CPU & Memory Metrics")
print("="*80)

if isinstance(cpu_mem, dict) and "error" not in cpu_mem:
    has_data = False
    
    # CPU Utilization
    if cpu_mem.get("cpu_utilization_percent") is not None:
        print(f"  CPU Utilization: {cpu_mem['cpu_utilization_percent']:.1f}%")
        has_data = True
    
    # Memory
    if cpu_mem.get("memory_total_bytes") and cpu_mem.get("memory_used_bytes"):
        total_gb = cpu_mem["memory_total_bytes"] / (1024**3)
        used_gb = cpu_mem["memory_used_bytes"] / (1024**3)
        free_gb = cpu_mem.get("memory_free_bytes", 0) / (1024**3)
        pct = (cpu_mem["memory_used_bytes"] / cpu_mem["memory_total_bytes"]) * 100
        print(f"  Memory: {used_gb:.2f} GB / {total_gb:.2f} GB ({pct:.1f}% used, {free_gb:.2f} GB free)")
        has_data = True
    
    if not has_data:
        print("  No CPU/Memory metrics available from Triton")

elif isinstance(cpu_mem, dict) and "error" in cpu_mem:
    print(f"  Error fetching CPU/Memory metrics: {cpu_mem.get('error')}")
    if cpu_mem.get("detail"):
        print(f"  Detail: {cpu_mem.get('detail')}")
else:
    print("  No CPU/Memory metrics available")

In [ ]:
# Get inference metrics per model from the proxy
inference_metrics = get_inference_metrics()

print("\n" + "="*80)
print(" Inference Metrics (per model)")
print("="*80)

if isinstance(inference_metrics, list) and inference_metrics:
    for model in inference_metrics:
        model_name = model.get("model", "unknown")
        version = model.get("version")
        
        print(f"\n  Model: {model_name}" + (f" (v{version})" if version else ""))
        
        if model.get("inference_count"):
            print(f"    Inference Count: {model['inference_count']}")
        if model.get("inference_success"):
            print(f"    Successful Requests: {model['inference_success']}")
        if model.get("inference_failure"):
            print(f"    Failed Requests: {model['inference_failure']}")
        if model.get("avg_request_duration_ms") is not None:
            print(f"    Avg Request Duration: {model['avg_request_duration_ms']:.2f} ms")
        if model.get("avg_queue_duration_ms") is not None:
            print(f"    Avg Queue Time: {model['avg_queue_duration_ms']:.2f} ms")
        if model.get("avg_compute_duration_ms") is not None:
            print(f"    Avg Compute Time: {model['avg_compute_duration_ms']:.2f} ms")

elif isinstance(inference_metrics, dict) and "error" in inference_metrics:
    print(f"  Error fetching inference metrics: {inference_metrics.get('error')}")
    if inference_metrics.get("detail"):
        print(f"  Detail: {inference_metrics.get('detail')}")
else:
    print("  No inference metrics available (no models have been invoked yet)")

In [ ]:
# Get all parsed metrics (uncomment to see summary)

# parsed = get_parsed_metrics()
# if "error" not in parsed:
#     print(f"GPU count: {len(parsed.get('gpu', []))}")
#     print(f"CPU/Memory data: {'Yes' if parsed.get('cpu_memory') else 'No'}")
#     print(f"Models with inference data: {len(parsed.get('models', []))}")
#     print(f"Raw metric count: {parsed.get('raw_metric_count', 0)}")
# else:
#     print(f"Error: {parsed.get('error')}")

# Get raw Prometheus metrics (uncomment to see raw data)

# raw = get_raw_metrics()
# if not raw.startswith("Error:"):
#     # Show first 50 lines
#     lines = raw.strip().split('\n')[:50]
#     print(f"Total lines: {len(raw.strip().split(chr(10)))}")
#     print("\nFirst 50 lines:")
#     for line in lines:
#         print(line)
# else:
#     print(raw)

---
# Summary

This notebook provides the following administrative capabilities:

| Function | Description |
|----------|-------------|
| `scale_deployment(replicas)` | Scale the Triton deployment to N replicas |
| `restart_deployment()` | Trigger a rolling restart |
| `get_deployment_status()` | Get deployment and pod status |
| `get_deployment_resources()` | View current CPU/Memory/GPU configuration |
| `update_deployment_resources(...)` | Update resource requests/limits |
| `list_models()` | List all models with status |
| `load_model(name)` | Load a model into memory |
| `unload_model(name)` | Unload a model from memory |
| `pin_model(name)` | Pin model to prevent eviction |
| `unpin_model(name)` | Unpin model to allow eviction |
| `cordon_model(name)` | Stop model from accepting requests |
| `uncordon_model(name)` | Resume model accepting requests |
| `get_gpu_metrics()` | Get GPU memory, utilization, power, and temperature |
| `get_cpu_memory_metrics()` | Get CPU utilization and system memory usage |
| `get_inference_metrics()` | Get per-model inference counts and latencies |
| `get_parsed_metrics()` | Get all metrics in structured format |
| `get_raw_metrics()` | Get raw Prometheus metrics text |

## Endpoints Used

| Endpoint | Description |
|----------|-------------|
| `{proxy_url}/v1/dashboard/models` | Model list and status |
| `{proxy_url}/v1/models/{name}/pin` | Pin/unpin models |
| `{proxy_url}/v1/models/{name}/cordon` | Cordon models |
| `{proxy_url}/v1/models/{name}/uncordon` | Uncordon models |
| `{proxy_url}/v2/repository/models/{name}/load` | Load models |
| `{proxy_url}/v2/repository/models/{name}/unload` | Unload models |
| `{proxy_url}/v1/metrics` | Raw Prometheus metrics |
| `{proxy_url}/v1/metrics/parsed` | All metrics parsed |
| `{proxy_url}/v1/metrics/gpu` | GPU metrics |
| `{proxy_url}/v1/metrics/cpu` | CPU/memory metrics |
| `{proxy_url}/v1/metrics/inference` | Per-model inference metrics |
| `{admin_url}/v1/deployments/...` | Kubernetes deployment management |